In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Refutational Completeness of the Cut Rule

This notebook implements a number of procedures that are needed in our proof of the <em style="color:blue">refutational completeness</em> of the cut rule.

We use the types `Literal`, `Clause`, and `CNF` that are defined in the notebook `04-CNF.ipynb`:
```
type Literal =
    | Pos of string   // p
    | Neg of string   // ¬p

type Clause = Set<Literal>
type CNF    = Set<Clause>
```

In [2]:
importNotebook "04-CNF.ipynb"

imported 

Propositional-Logic-Parser.ipynb, 04-CNF.ipynb

The function `complement` takes one argument:
- `l` is a literal.

It returns the *complement* of `l`.
As we are only dealing with formulas in CNF that are written in set notation and these formulas do not contain $\top$ or $\bot$, a literal is either 
a propositional variable or the negation of a propositional variable.
Therefore, if $p$ is a propositional variable, we have the following: 
* $\texttt{complement}(p) = \neg p$,
* $\texttt{complement}(\neg p) = p$.

In [3]:
let complement (l: Literal) : Literal =
    match l with
    | Pos p -> Neg p
    | Neg p -> Pos p

In [4]:
complement (Pos "p")

Neg "p"

In [5]:
complement (Neg "p")

Pos "p"

The function `extractVariable` takes one argument:
- `l` is a literal.

It returns the propositional variable of the literal `l`.
If $p$ is a propositional variable, we have the following: 
<ol>
    <li>$\texttt{extractVariable}(p) = p$,
    </li>
    <li>$\texttt{extractVariable}(\neg p) = p$.
    </li>
</ol>

In [6]:
let extractVariable (l: Literal) : string =
    match l with
    | Pos p -> p
    | Neg p -> p

In [7]:
extractVariable (Pos "p")

"p"

In [8]:
extractVariable (Neg "q")

"q"

The function `collectVariables` takes one argument:
- `M` is a set of clauses.

It returns the set of all propositional variables occurring in $M$.

In [9]:
let collectVariables (M: CNF) : Set<string> =
    set [ for C in M do for l in C -> extractVariable l ]

In [10]:
let C1 : Clause = set [ Pos "p"; Pos "q"; Pos "r" ]
let C2 : Clause = set [ Neg "p"; Neg "q"; Neg "s" ]
collectVariables (set [ C1; C2 ])

set ["p"; "q"; "r"; "s"]

The function `cutRule` takes two arguments:
- `C1` is a clause,
- `C2` is a clause.

It returns the set of all clauses that can be derived from $C_1$ and $C_2$ using the *cut rule*.  In set notation, the cut rule is the following rule of inference:
$$
   \frac{\displaystyle \;C_1\cup \{l\} \quad C_2 \cup \bigl\{\overline{\,l\,}\bigr\}}{\displaystyle C_1 \cup C_2}
$$
The operator `-` computes the difference of two sets and `+` computes their union.

In [11]:
let cutRule (C1: Clause) (C2: Clause) : Set<Clause> =
    set [ for l in C1 do
            if C2.Contains (complement l) then
                (C1 - set [ l ]) + (C2 - set [ complement l ]) ]

In [12]:
cutRule (set [ Neg "q"; Neg "r" ]) (set [ Pos "q"; Pos "r" ])

set [set [Pos "q"; Neg "q"]; set [Pos "r"; Neg "r"]]

The function `saturate` takes one argument:
- `Clauses` is a set of clauses.

It returns the set of all clauses that can be derived from clauses in the set `Clauses` using the *cut rule*.  The function keeps applying the cut rule until either no new clauses can be derived, or the empty clause $\{\}$ is derived.  In the latter case, the set $\bigl\{\{\}\bigr\}$ is returned.  The resulting set of clauses is *saturated* in the following sense:  If $C_1$ and $C_2$ are clauses from the resulting set and the clause $D$ can be derived from $C_1$ and $C_2$ via the cut rule, then $D$ is also an element of the resulting set, or the resulting set is $\bigl\{\{\}\bigr\}$.

**Implementation:** The function is implemented as a recursive function: Every recursive call corresponds to one iteration of the loop in the Python implementation.  The set `Derived` contains all clauses that can be derived from two clauses in `Clauses`.  If `Derived` contains no new clauses, `Clauses` is saturated.  Otherwise, the new clauses are added and the process is repeated.

In [13]:
let rec saturate (Clauses: CNF) : CNF =
    let Derived = set [ for C1 in Clauses do
                          for C2 in Clauses do
                            yield! cutRule C1 C2 ]
    if Derived.Contains Set.empty then
        set [ Set.empty ]               // This is the set notation of ⊥.
    else
        let New = Derived - Clauses     // Remove clauses that were already present before.
        if New.IsEmpty then             // No new clauses have been found.
            Clauses
        else
            saturate (Clauses + New)

In [14]:
let D1 : Clause = set [ Pos "p"; Pos "q" ]
let D2 : Clause = set [ Neg "p" ]
let D3 : Clause = set [ Neg "p"; Neg "q" ]
saturate (set [ D1; D2; D3 ])

set [set [Pos "p"; Pos "q"]; set [Pos "p"; Neg "p"]; set [Pos "q"]; set [Pos "q"; Neg "q"]; set [Neg "p"]; set [Neg "p"; Neg "q"]]

The function `findValuation` takes one argument:
- `Clauses` is a set of clauses.

The function tries to compute a variable interpretation that makes all of these clauses `true`.  The result has the type `Set<Literal> option`:
- If `Clauses` is satisfiable, `Some Literals` is returned.  Here, `Literals` is a set of literals that does not contain any complementary literals and therefore corresponds to a variable assignment satisfying all clauses.
- If `Clauses` is unsatisfiable, `None` is returned.

**Implementation:** First, the set of clauses is saturated.  If the saturated set contains the empty clause, `Clauses` is unsatisfiable.  Otherwise, the variables are processed one by one.  The mutable variable `Literals` contains the literals that have been chosen so far.  For every variable $p$ we check, whether there is a clause $C$ in the saturated set such that 
- $p \in C$ and 
- all other literals of $C$ are already false, i.e. their complements are elements of `Literals`.

In this case, $p$ has to be true.  Otherwise, we can safely set $p$ to false.  The expression `A.IsSubsetOf B` checks whether $A \subseteq B$.

In [15]:
let findValuation (Clauses: CNF) : Set<Literal> option =
    let Variables = collectVariables Clauses
    let Clauses   = saturate Clauses
    if Clauses.Contains Set.empty then  // The set Clauses is inconsistent.
        None
    else
        let mutable Literals : Set<Literal> = Set.empty
        for p in Variables do
            let Complements = Set.map complement Literals
            if Clauses |> Set.exists (fun C -> C.Contains (Pos p) && 
                                               (C - set [ Pos p ]).IsSubsetOf Complements) then
                Literals <- Literals.Add (Pos p)
            else
                Literals <- Literals.Add (Neg p)
        Some Literals

In [16]:
let C01 : Clause = set [ Pos "r"; Pos "p"; Pos "s" ]
let C02 : Clause = set [ Pos "r"; Pos "s" ]
let C03 : Clause = set [ Pos "p"; Pos "q"; Pos "s" ]
let C04 : Clause = set [ Neg "p"; Neg "q" ]
let C05 : Clause = set [ Neg "p"; Pos "s"; Neg "r" ]
let C06 : Clause = set [ Pos "p"; Neg "q"; Pos "r" ]
let C07 : Clause = set [ Neg "r"; Neg "s"; Pos "q" ]
let C08 : Clause = set [ Neg "p"; Neg "s" ]
let C09 : Clause = set [ Pos "p"; Neg "r"; Neg "q" ]
let C10 : Clause = set [ Neg "p"; Pos "r"; Pos "q"; Neg "s" ]
let C11 : Clause = set [ Pos "p"; Pos "q"; Pos "r"; Neg "s" ]

In [17]:
findValuation (set [ C01; C02; C03; C04; C05; C06; C07; C08; C09; C10 ])

Some set [Pos "s"; Neg "p"; Neg "q"; Neg "r"]

In [18]:
let Clauses = set [ C01; C02; C03; C04; C05; C06; C07; C08; C09; C10; C11 ]
findValuation Clauses

None

In [19]:
saturate Clauses

set [set []]